In [ ]:
import hddm
import pandas as pd
from pathlib import Path
from typing import List

# ==============================================================================
# Post-Hoc Model Comparison Script
# Description: Extracts Deviance Information Criterion (DIC) from a suite of 
#              fitted HDDM models to evaluate relative goodness-of-fit.
# ==============================================================================

def compare_model_dic(
    output_dir: Path, 
    save_dir: Path, 
    model_names: List[str]
) -> pd.DataFrame:
    """
    Iterates through a list of fitted HDDM models, extracts their DIC values, 
    computes relative fit metrics (Delta DIC), and exports the summary table.

    Args:
        output_dir (Path): Base directory containing the fitted model files.
        save_dir (Path): Target directory for saving the comparison results.
        model_names (List[str]): List of model identifiers to evaluate.

    Returns:
        pd.DataFrame: A sorted dataframe containing model names, raw DIC, 
        and Delta DIC values. Returns an empty dataframe if no models are found.
    """
    # Ensure the output directory exists
    save_dir.mkdir(parents=True, exist_ok=True)
    
    dic_results = []
    
    print("\n[INFO] Initiating DIC extraction for model comparison...")

    for model_name in model_names:
        # Assuming the model was saved as a directory or a direct .model file
        # Adjust the path resolution if your save format requires a suffix 
        # (e.g., model_path = output_dir / model_name / f"{model_name}.model")
        model_path = output_dir / model_name
        
        if model_path.exists():
            print(f"[INFO] Loading model: {model_name} ...")
            try:
                # Load the serialized HDDM model object
                m = hddm.load(str(model_path))
                
                # Extract the Deviance Information Criterion (DIC)
                dic_results.append({
                    "model_name": model_name,
                    "dic": float(m.dic)
                })
            except Exception as e:
                print(f"[ERROR] Failed to load or extract DIC for {model_name}. Exception: {e}")
        else:
            print(f"[WARNING] Skipping {model_name}: File or directory not found at {model_path}")

    # Aggregate and compute comparative fit indices
    if dic_results:
        df_dic = pd.DataFrame(dic_results)
        
        # Sort models by DIC (lower values indicate a better penalized model fit)
        df_dic = df_dic.sort_values("dic").reset_index(drop=True)
        
        # Compute Delta DIC relative to the best-fitting (lowest DIC) model in the set
        df_dic["delta_dic"] = df_dic["dic"] - df_dic["dic"].min()
        
        # Export the sorted comparison table for manuscript reporting
        save_path = save_dir / "manual_model_comparison_dic_2.0.csv"
        df_dic.to_csv(save_path, index=False)
        
        print(f"\n[SUCCESS] Extraction complete. Comparison table saved to: {save_path}")
        return df_dic
    else:
        print("\n[ERROR] No DIC values extracted. Please verify directory paths and model names.")
        return pd.DataFrame()


if __name__ == "__main__":
    # 1. Define base input and output directories
    base_output_dir = Path("../../reference_outputs/hddm/formal_models")
    target_save_dir = Path("output/result/formal_model")

    # 2. Define the exact namespace of the models to be compared
    models_to_evaluate = [
        "M0_baseline",
        "M1_v_train_main",
        "M2_a_train_main",
        "M3_av_train_main",
        "M4_v_judgment_x_pair",
        "M5_a_judgment_x_pair",
        "M6_v_interaction_a_train",
        "M7_a_interaction_v_train",
        "M8_av_judgment_x_pair",
        # Append additional model names here if necessary...
    ]

    # 3. Execute the comparison extraction
    comparison_df = compare_model_dic(
        output_dir=base_output_dir, 
        save_dir=target_save_dir, 
        model_names=models_to_evaluate
    )
    
    # 4. Display the formatted dataframe in Jupyter Notebook (if applicable)
    if not comparison_df.empty:
        # Note: 'display()' is a built-in function in IPython/Jupyter environments.
        # If running via a standard Python terminal, you may want to use 'print(comparison_df)' instead.
        display(comparison_df)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path  # Added missing import for Path instantiation

# ==============================================================================
# 1. Configuration & Setup
# ==============================================================================
base_directory = "../../reference_outputs/hddm/formal_models"
save_dir = Path("output/result/formal_model")

# Specify the target models to be included in the visualization
top_models = [
    # "M0_baseline",
    # "M1_v_train_main",
    # "M2_a_train_main",
    # "M3_av_train_main",
    "M4_v_judgment_x_pair",
    # "M5_a_judgment_x_pair",
    # "M6_v_interaction_a_train",
    # "M7_a_interaction_v_train",
    # "M8_av_judgment_x_pair"
]

# Keywords to isolate specific interaction or main effect parameters
target_keywords = [":", "T.continuous", "T.within"] 

print("==================================================")
print("[INFO] Initiating flattened plotting script for posterior estimates...")
print("==================================================")

records = []

# ==============================================================================
# 2. Data Extraction
# ==============================================================================
for model_name in top_models:
    # Construct expected path for posterior stats
    stats_file = f"{base_directory}/{model_name}_posterior_stats.csv"
    
    # Verify file existence before processing
    if not os.path.exists(stats_file):
        print(f"[WARNING] File not found: {stats_file}. Skipping.")
        continue
    else:
        print(f"[INFO] Successfully located: {stats_file}")
        
    # Load posterior statistics
    df_stats = pd.read_csv(stats_file, index_col=0)
    
    found_params_count = 0
    for param in df_stats.index:
        param_str = str(param)
        
        # Filter out subject-level parameters, standard deviations, and base cognitive parameters
        if "_subj" in param_str or "std" in param_str or param_str in ['a', 'v', 't', 'z']:
            continue
            
        # Extract target parameters matching predefined keywords
        is_target = any(kw in param_str for kw in target_keywords)
        if is_target:
            records.append({
                "Model": model_name,
                "Parameter": param_str,
                "Mean": float(df_stats.loc[param, "mean"]),
                "Lower": float(df_stats.loc[param, "2.5q"]),
                "Upper": float(df_stats.loc[param, "97.5q"])
            })
            found_params_count += 1
            
    print(f"       -> Extracted {found_params_count} core parameters from {model_name}.")

# ==============================================================================
# 3. Visualization Preparation
# ==============================================================================
if len(records) == 0:
    print("\n[ERROR] Fatal: No data extracted. Please verify directory paths and target_keywords.")
else:
    print("\n[INFO] Constructing visualization data structures...")
    df_plot = pd.DataFrame(records)
    
    unique_params = sorted(df_plot["Parameter"].unique())
    print(f"[INFO] Identified {len(unique_params)} unique parameters for comparison.")
    
    param_to_y = {p: i for i, p in enumerate(unique_params)}
    y_positions = np.arange(len(unique_params))
    
    # Dynamically scale figure height based on the number of parameters
    fig, ax = plt.subplots(figsize=(10, max(5, len(unique_params) * 1.5)))
    colors = plt.cm.tab10(np.linspace(0, 1, max(10, len(top_models))))
    
    # ==============================================================================
    # 4. Plot Execution
    # ==============================================================================
    print("[INFO] Rendering Highest Density Interval (HDI) error bars...")
    for i, model in enumerate(top_models):
        sub_df = df_plot[df_plot["Model"] == model]
        if sub_df.empty: 
            continue
            
        # Calculate dodge offset to prevent overlapping error bars
        offset = (i - len(top_models)/2 + 0.5) * 0.15 
        
        # Flag to ensure the model label is added to the legend only once
        added_label = False 
        
        for _, row in sub_df.iterrows():
            y_base = param_to_y[row["Parameter"]]
            y = y_base + offset
            
            x_mean = row["Mean"]
            xerr_lower = x_mean - row["Lower"]
            xerr_upper = row["Upper"] - x_mean
            
            ax.errorbar(
                x_mean, y, xerr=[[xerr_lower], [xerr_upper]], 
                fmt='o', color=colors[i], capsize=4, elinewidth=2, markersize=8,
                label=model if not added_label else ""  
            )
            added_label = True 

    # Align y-ticks and simplify categorical treatment labels for cleaner visualization
    ax.set_yticks(y_positions)
    clean_labels = [
        p.replace("Treatment('categorical')", "cat").replace("Treatment('between')", "btw") 
        for p in unique_params
    ]
    ax.set_yticklabels(clean_labels, fontsize=10)
    
    # Add vertical reference line at zero
    ax.axvline(0, color='gray', linestyle='--', linewidth=1.5, alpha=0.8)
    
    # Set labels, title, and limits
    ax.set_xlabel("Posterior Estimate (Mean & 95% HDI)", fontsize=12)
    ax.set_title("Comparison of 95% HDI Across Selected Models", fontsize=14, pad=15)
    ax.set_ylim(-1, len(unique_params))
    
    # Configure legend and grid
    ax.legend(title="Models", bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(axis='x', linestyle=':', alpha=0.6)
    
    plt.tight_layout()
    
    # Ensure output directory exists
    save_dir.mkdir(parents=True, exist_ok=True)
    
    # Construct safe filename and export
    model_names_str = "_".join([m.replace("M", "") for m in top_models])
    save_path = save_dir / f"model_comparison_{model_names_str}.png"
    
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"[SUCCESS] Figure exported successfully to: {save_path}")
    
    # Display the final plot
    plt.show()
    print("[SUCCESS] Visualization complete.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ==========================================
# 1. 设置区 (请确保路径和模型名称和你的文件夹一致)
# ==========================================
base_directory = "../../reference_outputs/hddm/formal_models"
save_directory = "output/result/formal_model"
top_models = [
    #"M0_baseline",
    #"M1_v_train_main",
    #"M2_a_train_main",
    #"M3_av_train_main",
    "M4_v_judgment_x_pair",
   # "M5_a_judgment_x_pair",
    #"M6_v_interaction_a_train",
    #"M7_a_interaction_v_train",
    #"M8_av_judgment_x_pair"
]
target_keywords = [":", "T.continuous", "T.within"] 

print("==================================================")
print("🚀 开始为每个模型单独绘制参数后验图...")
print("==================================================")

# ==========================================
# 2. 循环处理每个模型：读取数据 + 独立绘图
# ==========================================
for model_name in top_models:
    # 拼凑路径
    stats_file = f"{base_directory}/{model_name}_posterior_stats.csv"
    
    # 检查文件是否存在
    if not os.path.exists(stats_file):
        print(f"❌ 警告: 找不到文件 {stats_file}")
        continue
    else:
        print(f"✅ 成功找到: {stats_file}")
        
    # 读取数据
    df_stats = pd.read_csv(stats_file, index_col=0)
    
    # 当前模型的参数记录
    records = []
    found_params_count = 0
    
    for param in df_stats.index:
        param_str = str(param)
        
        # 过滤不需要的参数
        if "_subj" in param_str or "std" in param_str or param_str in ['a', 'v', 't', 'z']:
            continue
            
        is_target = any(kw in param_str for kw in target_keywords)
        if is_target:
            records.append({
                "Parameter": param_str,
                "Mean": float(df_stats.loc[param, "mean"]),
                "Lower": float(df_stats.loc[param, "2.5q"]),
                "Upper": float(df_stats.loc[param, "97.5q"])
            })
            found_params_count += 1
            
    print(f"   -> 在 {model_name} 中提取了 {found_params_count} 个核心参数")

    # ==========================================
    # 3. 绘图执行区 (针对当前模型)
    # ==========================================
    if len(records) == 0:
        print(f"   -> ⚠️ 警告：{model_name} 没有提取到任何目标参数，跳过绘图。\n")
        continue

    print(f"   -> 🎨 正在为 {model_name} 绘制误差棒...")
    df_plot = pd.DataFrame(records)
    
    unique_params = sorted(df_plot["Parameter"].unique())
    param_to_y = {p: i for i, p in enumerate(unique_params)}
    y_positions = np.arange(len(unique_params))
    
    # 动态调整画布高度
    fig, ax = plt.subplots(figsize=(8, max(4, len(unique_params) * 0.8)))
    
    # 绘制当前模型的所有提取参数
    for _, row in df_plot.iterrows():
        y = param_to_y[row["Parameter"]]
        
        x_mean = row["Mean"]
        xerr_lower = x_mean - row["Lower"]
        xerr_upper = row["Upper"] - x_mean
        
        ax.errorbar(
            x_mean, y, xerr=[[xerr_lower], [xerr_upper]], 
            fmt='o', color='steelblue', capsize=4, elinewidth=2, markersize=8
        )

    ax.set_yticks(y_positions)
    # 简化标签显示
    # ==========================================
    # 🌟 核心修改：基于实验设计提取标准主效应/交互作用名称
    # ==========================================
    clean_labels = []
    for p in unique_params:
        # 1. 提取是哪个参数 (v 还是 a)
        prefix = "v" if p.startswith("v") else "a" if p.startswith("a") else "Param"
        
        # 2. 根据关键字直接映射为标准的实验设计因子
        if ":" in p:
            body = "Train Type × Pair Type"
        elif "continuous" in p.lower():
            body = "Train Type"
        elif "within" in p.lower():
            body = "Pair Type"
        else:
            body = p # 兜底
            
        clean_labels.append(f"{prefix}: {body}")
        
    # 应用新标签，适当加粗字体
    ax.set_yticklabels(clean_labels, fontsize=12, fontweight='bold')

    
    # 添加 0 基准线
    ax.axvline(0, color='gray', linestyle='--', linewidth=1.5, alpha=0.8)
    
    ax.set_xlabel("Posterior Estimate (Mean & 95% HDI)", fontsize=12)
    ax.set_title(f"Posterior Estimates: {model_name}", fontsize=14, pad=15)
    
    # 调整 Y 轴显示范围使其留有合适的边距
    ax.set_ylim(-0.5, len(unique_params) - 0.5)
    
    ax.grid(axis='x', linestyle=':', alpha=0.6)
    plt.tight_layout()
    
    # 保存图片到模型文件夹中
    save_path = f"{save_directory}/{model_name}/{model_name}_posterior_plot.png"
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show() # 在 Jupyter 环境中直接显示图片
    plt.close() # 画完关闭，防止内存泄漏
    
    print(f"✨ {model_name} 画图完成！已保存至 {save_path}\n")

print("==================================================")
print("🎉 所有模型的绘图任务已全部结束！")